# Data Splitting: In-Sample vs Out-of-Sample

This notebook splits the preprocessed oil data into:
- **In-Sample**: Data before 2018 (for training)
- **Out-of-Sample**: Data from 2018 onwards (for testing/validation)

**Target Variable**: `cl1_ret` (crude oil returns)

## Setup

Load required libraries and read the preprocessed data.

In [15]:
from pathlib import Path
import pandas as pd
import numpy as np

# Input file from preprocessing pipeline
INPUT_FILE = Path("Oil_Model_Weekly_Core.csv")

# Output files
OUT_INSAMPLE = Path("Oil_InSample.csv")
OUT_OUTSAMPLE = Path("Oil_OutOfSample.csv")

## Load Data

Read the model-ready dataset and inspect it.

In [16]:
# Load data with date index
df = pd.read_csv(INPUT_FILE, index_col=0, parse_dates=True)

print(f"Full dataset shape: {df.shape}")
print(f"Date range: {df.index.min().date()} → {df.index.max().date()}")
print(f"\nTarget variable (cl1_ret) statistics:")
print(df['cl1_ret'].describe())
print(f"\nFirst few rows:")
print(df.head())

Full dataset shape: (1534, 26)
Date range: 1995-02-10 → 2024-06-28

Target variable (cl1_ret) statistics:
count    1534.000000
mean        0.000957
std         0.053066
min        -0.346863
25%        -0.026759
50%         0.004757
75%         0.030161
max         0.275756
Name: cl1_ret, dtype: float64

First few rows:
             cl1_ret  cl1_ret_l1  cl1_ret_l2  cl1_ret_l3  cl1_ret_l4  \
date                                                                   
1995-02-10 -0.017186    0.045202   -0.038256    0.062503   -0.008525   
1995-02-17  0.024085   -0.017186    0.045202   -0.038256    0.062503   
1995-02-24 -0.011702    0.024085   -0.017186    0.045202   -0.038256   
1995-03-03 -0.003215   -0.011702    0.024085   -0.017186    0.045202   
1995-03-10 -0.039414   -0.003215   -0.011702    0.024085   -0.017186   

            brent_ret   cl2_ret  bdiy_index_ret   spx_ret  dxy_returns  ...  \
date                                                                    ...   
1995-02-10  -0.0

## Split Data: 2018 Cutoff

- **In-Sample**: < 2018 (training data)
- **Out-of-Sample**: >= 2018 (testing data)

In [17]:
# Define cutoff date
cutoff_date = pd.Timestamp('2018-01-01')

# Split data
insample = df[df.index < cutoff_date].copy()
outsample = df[df.index >= cutoff_date].copy()

print(f"Split cutoff: {cutoff_date.date()}\n")
print(f"In-Sample (< 2018):")
print(f"  Rows: {len(insample):,}")
print(f"  Date range: {insample.index.min().date()} → {insample.index.max().date()}")
print(f"  Target (cl1_ret) mean: {insample['cl1_ret'].mean():.6f}")
print(f"  Target (cl1_ret) std:  {insample['cl1_ret'].std():.6f}")

print(f"\nOut-of-Sample (>= 2018):")
print(f"  Rows: {len(outsample):,}")
print(f"  Date range: {outsample.index.min().date()} → {outsample.index.max().date()}")
print(f"  Target (cl1_ret) mean: {outsample['cl1_ret'].mean():.6f}")
print(f"  Target (cl1_ret) std:  {outsample['cl1_ret'].std():.6f}")

print(f"\nSplit ratio:")
print(f"  In-Sample: {len(insample)/(len(insample)+len(outsample))*100:.1f}%")
print(f"  Out-of-Sample: {len(outsample)/(len(insample)+len(outsample))*100:.1f}%")

Split cutoff: 2018-01-01

In-Sample (< 2018):
  Rows: 1,195
  Date range: 1995-02-10 → 2017-12-29
  Target (cl1_ret) mean: 0.000978
  Target (cl1_ret) std:  0.050351

Out-of-Sample (>= 2018):
  Rows: 339
  Date range: 2018-01-05 → 2024-06-28
  Target (cl1_ret) mean: 0.000884
  Target (cl1_ret) std:  0.061774

Split ratio:
  In-Sample: 77.9%
  Out-of-Sample: 22.1%


## Data Quality Check

Verify no missing values and check feature alignment.

In [18]:
# Check missing values
print("In-Sample missing values:")
print(f"  Total: {insample.isna().sum().sum()}")

print("\nOut-of-Sample missing values:")
print(f"  Total: {outsample.isna().sum().sum()}")

# Check target variable presence
print(f"\n'cl1_ret' present in both splits: {('cl1_ret' in insample.columns) and ('cl1_ret' in outsample.columns)}")

# Check feature consistency
print(f"\nFeatures in both splits:")
common_features = list(set(insample.columns) & set(outsample.columns))
print(f"  Total: {len(common_features)}")
print(f"  Features: {sorted(common_features)[:5]}...")  # Show first 5

In-Sample missing values:
  Total: 0

Out-of-Sample missing values:
  Total: 0

'cl1_ret' present in both splits: True

Features in both splits:
  Total: 26
  Features: ['10y_yield_diff', '3m_yield_diff', 'bdiy_index_ret', 'bp_ret', 'brent_ret']...


## Save Split Datasets

Export in-sample and out-of-sample data to CSV files.

In [19]:
# Save splits
insample.to_csv(OUT_INSAMPLE)
outsample.to_csv(OUT_OUTSAMPLE)

print(f"✓ In-Sample saved: {OUT_INSAMPLE.resolve()}")
print(f"✓ Out-of-Sample saved: {OUT_OUTSAMPLE.resolve()}")

print(f"\n" + "="*60)
print("Data split complete!")
print("="*60)

✓ In-Sample saved: /Users/faisal/Desktop/Econometrics/cw1/Oil_InSample.csv
✓ Out-of-Sample saved: /Users/faisal/Desktop/Econometrics/cw1/Oil_OutOfSample.csv

Data split complete!


## Summary

**Target Variable**: `cl1_ret` (WTI crude oil returns)

**In-Sample (Training)**:
- Used to fit/train forecasting models
- Covers historical data before 2018

**Out-of-Sample (Testing)**:
- Used to evaluate model performance
- Covers recent data from 2018 onwards
- Represents "unseen" future data to the model

## Inspect Split Data

Read and display first and last rows of both datasets.

In [11]:
# Read the saved split files
insample_loaded = pd.read_csv(OUT_INSAMPLE, index_col=0, parse_dates=True)
outsample_loaded = pd.read_csv(OUT_OUTSAMPLE, index_col=0, parse_dates=True)

print("="*80)
print("IN-SAMPLE DATA (< 2018)")
print("="*80)
print(f"\nShape: {insample_loaded.shape}\n")

print("HEAD (first 5 rows):")
print(insample_loaded.head())

print("\n" + "-"*80 + "\n")

print("TAIL (last 5 rows):")
print(insample_loaded.tail())

print("\n" + "="*80)
print("OUT-OF-SAMPLE DATA (>= 2018)")
print("="*80)
print(f"\nShape: {outsample_loaded.shape}\n")

print("HEAD (first 5 rows):")
print(outsample_loaded.head())

print("\n" + "-"*80 + "\n")

print("TAIL (last 5 rows):")
print(outsample_loaded.tail())

IN-SAMPLE DATA (< 2018)

Shape: (1195, 26)

HEAD (first 5 rows):
             cl1_ret  cl1_ret_l1  cl1_ret_l2  cl1_ret_l3  cl1_ret_l4  \
date                                                                   
1995-02-10 -0.017186    0.045202   -0.038256    0.062503   -0.008525   
1995-02-17  0.024085   -0.017186    0.045202   -0.038256    0.062503   
1995-02-24 -0.011702    0.024085   -0.017186    0.045202   -0.038256   
1995-03-03 -0.003215   -0.011702    0.024085   -0.017186    0.045202   
1995-03-10 -0.039414   -0.003215   -0.011702    0.024085   -0.017186   

            brent_ret   cl2_ret  bdiy_index_ret   spx_ret  dxy_returns  ...  \
date                                                                    ...   
1995-02-10  -0.000615  0.033584       -0.016824  0.023988    -0.008262  ...   
1995-02-17   0.022514 -0.009247       -0.016097  0.000000    -0.008262  ...   
1995-02-24   0.017298  0.023225       -0.002539  0.000000    -0.008262  ...   
1995-03-03  -0.031234 -0.012896    